# Databricks Solutions Architect Masterclass
## Data Engineering on Databricks: From Raw Ingestion to High-Performance Analytics

**Duration**: Comprehensive learning path  
**Level**: Intermediate to Advanced  
**Focus**: Production-grade data engineering patterns on Databricks

---

## Masterclass Outline

1. **Delta Lake Essentials** - Foundation of reliable data
2. **Medallion Architecture** - Layered data organization
3. **Performance Tuning** - Optimizing at scale
4. **Databricks Features** - Governance and collaboration tools
5. **PySpark Mastery** - DataFrame API and advanced patterns
6. **Advanced Optimization** - Z-Ordering and Data Skipping
7. **End-to-End Workloads** - Real-world scenarios

## Section 1: Delta Lake Essentials

### What is Delta Lake?
Delta Lake is an open-source storage framework that brings ACID transactions and time-travel capabilities to Apache Spark and Databricks. It provides:

- **ACID Transactions**: Atomic, Consistent, Isolated, Durable operations
- **Schema Enforcement**: Prevent data corruption from schema mismatches
- **Time Travel**: Query data as it was at any point in time
- **Unified Batch and Streaming**: Same data format for all workloads
- **Data Lineage**: Track data changes and transformations

### Core Components
- **Transaction Log**: Versioned record of all changes (stored in `_delta_log/`)
- **Parquet Files**: Columnar storage for efficient querying
- **Metadata**: Stored in JSON and Parquet formats

In [ ]:
# Initialize Spark session with Delta Lake
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import datetime

spark = SparkSession.builder \
    .appName("Databricks-Masterclass") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Set to high verbosity for learning
spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"App Name: {spark.sparkContext.appName}")

### 1.1 Creating Delta Tables with ACID Guarantees

Delta Lake automatically provides ACID properties. When you write data to a Delta table, all writes are atomic—either the entire write succeeds or fails completely.

In [ ]:
# Create sample customer data
from datetime import datetime, timedelta

customer_data = [
    (1, "Alice Johnson", "alice@example.com", "2024-01-15", 150000.00),
    (2, "Bob Smith", "bob@example.com", "2024-02-20", 200000.00),
    (3, "Carol Davis", "carol@example.com", "2024-03-10", 175000.00),
    (4, "David Wilson", "david@example.com", "2024-01-25", 120000.00),
    (5, "Eve Martinez", "eve@example.com", "2024-03-05", 195000.00),
]

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("lifetime_value", DoubleType(), True),
])

df_customers = spark.createDataFrame(customer_data, schema=schema)

# Write as Delta table (ACID guaranteed)
delta_path = "/tmp/delta/customers_gold"

# Mode options: append, overwrite, error, ignore
df_customers.write.format("delta") \
    .mode("overwrite") \
    .save(delta_path)

print(f"✓ Delta table created at {delta_path}")
print(f"Total records: {df_customers.count()}")

### 1.2 Time Travel: Query Data at Any Point in Time

Delta Lake maintains a complete history of all changes. You can:
- Query data **as of a specific version**
- Query data **as of a specific timestamp**
- Restore tables to previous states
- Audit data changes

In [ ]:
# First, let's check the current version
print("=== Current Table State ===")
spark.sql(f"DESCRIBE DETAIL delta.`{delta_path}`").show()

# Display current data
print("\n=== Current Data ===")
spark.read.format("delta").load(delta_path).show()

In [ ]:
# Make a modification: Update Alice's lifetime value
import time

time.sleep(2)  # Small delay to ensure different timestamp

update_data = [(1, "Alice Johnson", "alice@example.com", "2024-01-15", 250000.00)]
df_update = spark.createDataFrame(update_data, schema=schema)

df_update.write.format("delta") \
    .mode("overwrite") \
    .save(delta_path)

print("✓ Data updated (Alice's lifetime value increased to 250000)")
print("\n=== New Data ===")
spark.read.format("delta").load(delta_path).show()

In [ ]:
# TIME TRAVEL: Query the previous version
print("=== Time Travel: VERSION AS OF 0 (Original Version) ===")
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(delta_path)

df_old.show()

# Show specific row
print("\nAlice's original lifetime value:")
df_old.filter(col("customer_id") == 1).select("name", "lifetime_value").show()

In [ ]:
# View transaction history
print("=== Delta Table History ===")
spark.sql(f"DESCRIBE HISTORY delta.`{delta_path}`").show(10, truncate=False)

### 1.3 Vacuum and Optimize: Maintenance Operations

**OPTIMIZE**: Compacts small files into larger ones for better query performance  
**VACUUM**: Removes old versions and deletes unused files to save storage

In [ ]:
# Check files before optimization
import os

print("=== Files Before Optimization ===")
# In real Databricks, use DBFS. For local, we'd check filesystem
# spark.sql(f"SELECT * FROM delta.`{delta_path}`").explain(True)

# OPTIMIZE: Compact files
print("\nOptimizing Delta table...")
spark.sql(f"OPTIMIZE delta.`{delta_path}`")

print("✓ Optimization complete")
print("\nOptimization removes small files and speeds up reads")

In [ ]:
# VACUUM: Remove old versions (default retention is 7 days)
# In production, this frees up storage

print("=== VACUUM Operation ===")
print("VACUUM removes files not referenced by the latest version")
print("Default retention: 7 days")

# Note: In some environments, you may need to set:
# spark.databricks.delta.retentionDurationCheck.enabled = false

try:
    spark.sql(f"VACUUM delta.`{delta_path}` RETAIN 0 HOURS")
    print("✓ Vacuum complete")
except:
    print("Note: VACUUM with retention 0 may be disabled for safety")
    print("Typical production retention: 7 days (default)")

### 1.4 Schema Enforcement and Evolution

Delta Lake prevents schema mismatches and allows controlled schema evolution.

In [ ]:
# Schema Enforcement: This will FAIL (different schema)
print("=== Schema Enforcement Test ===")

# Try to write data with wrong schema
wrong_schema_data = [(1, "Alice", "alice@example.com")]  # Missing fields
wrong_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
])

df_wrong = spark.createDataFrame(wrong_schema_data, schema=wrong_schema)

try:
    df_wrong.write.format("delta") \
        .mode("append") \
        .save(delta_path)
    print("Data written successfully")
except Exception as e:
    print(f"✓ Schema enforcement prevented the write: {str(e)[:100]}...")

In [ ]:
# Schema Evolution: ADD a new column safely
print("=== Schema Evolution ===")

# Add new column via mergeSchema
new_data = [
    (6, "Frank Brown", "frank@example.com", "2024-03-15", 165000.00, "premium"),
]

new_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("lifetime_value", DoubleType(), True),
    StructField("tier", StringType(), True),  # NEW COLUMN
])

df_new = spark.createDataFrame(new_data, schema=new_schema)

df_new.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .save(delta_path)

print("✓ Schema evolved: Added 'tier' column")
print("\n=== New Schema ===")
spark.read.format("delta").load(delta_path).printSchema()

---

## Section 2: Medallion Architecture

### The Medallion Architecture Pattern

The Medallion Architecture is a data design pattern that organizes data into three layers, each representing an increasing level of data quality and refinement:

```
┌─────────────────────────────────────────┐
│         GOLD LAYER (Analytics)          │
│  - Business-ready data                  │
│  - Aggregates, KPIs, Reports            │
│  - Optimized for BI tools               │
└─────────────────────────────────────────┘
                    ↑
┌─────────────────────────────────────────┐
│        SILVER LAYER (Refined)           │
│  - Deduplicated, cleaned data           │
│  - Business rules applied               │
│  - Multi-source consolidation           │
└─────────────────────────────────────────┘
                    ↑
┌─────────────────────────────────────────┐
│        BRONZE LAYER (Raw)               │
│  - As-is data ingestion                 │
│  - Minimal transformations              │
│  - Full data lineage preserved          │
└─────────────────────────────────────────┘
                    ↑
              External Data Sources
```

### Benefits
- **Data Quality**: Each layer ensures data quality standards
- **Reusability**: Silver layer serves multiple Gold layer tables
- **Scalability**: Easy to add new data consumers
- **Governance**: Clear data lineage and ownership
- **Performance**: Cached, optimized layers

### 2.1 Bronze Layer: Raw Data Ingestion

The Bronze layer ingests raw data with minimal transformation. Key characteristics:
- Schema-on-read (flexible)
- Includes metadata columns (ingestion_date, source, file_name)
- Serves as data archive/audit trail
- High volume, low quality checks

In [ ]:
# Simulate raw data from multiple sources
from datetime import datetime

# Source 1: API data
api_data = [
    (1, "Alice", "alice@example.com", 150000, datetime.now()),
    (2, "Bob", "bob@example.com", 200000, datetime.now()),
    (3, None, "carol@example.com", 175000, datetime.now()),  # Null value
]

# Source 2: CSV data with duplicates
csv_data = [
    (2, "Bob Smith", "bob@example.com", 200000, datetime.now()),  # Duplicate
    (4, "David", "david@example.com", 120000, datetime.now()),
    (5, "Eve", "eve@example.com", 195000, datetime.now()),
]

# Create DataFrames
bronze_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("lifetime_value", LongType(), True),
    StructField("loaded_at", TimestampType(), True),
])

df_api = spark.createDataFrame(api_data, schema=bronze_schema)
df_csv = spark.createDataFrame(csv_data, schema=bronze_schema)

# Add metadata
df_api = df_api.withColumn("source", lit("api")) \
    .withColumn("ingestion_timestamp", current_timestamp())

df_csv = df_csv.withColumn("source", lit("csv")) \
    .withColumn("ingestion_timestamp", current_timestamp())

# Combine all sources
df_bronze = df_api.union(df_csv)

# Write to Bronze layer
bronze_path = "/tmp/medallion/bronze/customers"
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .save(bronze_path)

print("=== Bronze Layer: Raw Data ===")
spark.read.format("delta").load(bronze_path).show(truncate=False)
print(f"\nTotal records in Bronze: {spark.read.format('delta').load(bronze_path).count()}")

### 2.2 Silver Layer: Data Cleaning & Consolidation

The Silver layer applies business logic:
- Remove duplicates
- Handle null values
- Standardize formats
- Validate data quality
- Track data lineage

In [ ]:
# Read from Bronze
df_bronze = spark.read.format("delta").load(bronze_path)

print("=== Bronze Layer Stats ===")
print(f"Total records: {df_bronze.count()}")
print(f"Null values in name: {df_bronze.filter(col('name').isNull()).count()}")
print(f"Null values in email: {df_bronze.filter(col('email').isNull()).count()}")
print(f"Duplicate customer_ids: {df_bronze.count() - df_bronze.select('customer_id').distinct().count()}")

In [ ]:
# Silver layer transformations
df_silver = df_bronze \
    .filter(col("name").isNotNull()) \
    .filter(col("email").isNotNull()) \
    .dropDuplicates(["customer_id"]) \
    .withColumn("name", trim(col("name"))) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("lifetime_value", col("lifetime_value").cast(DoubleType())) \
    .withColumn("data_quality_flag", lit("clean")) \
    .withColumn("processed_at", current_timestamp())

# Write to Silver layer
silver_path = "/tmp/medallion/silver/customers"
df_silver.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("=== Silver Layer: Cleaned Data ===")
spark.read.format("delta").load(silver_path).show(truncate=False)
print(f"\nTotal records in Silver: {spark.read.format('delta').load(silver_path).count()}")

### 2.3 Gold Layer: Business-Ready Analytics

The Gold layer creates aggregations and business metrics:
- Customer segments
- KPI calculations
- Dimensional tables
- Fact tables for analytics

In [ ]:
# Read from Silver
df_silver = spark.read.format("delta").load(silver_path)

# Create customer segments based on lifetime value
df_gold = df_silver \
    .withColumn("customer_segment", 
        when(col("lifetime_value") >= 200000, "Premium") \
        .when(col("lifetime_value") >= 150000, "Standard") \
        .otherwise("Basic")) \
    .withColumn("is_high_value", col("lifetime_value") >= 175000) \
    .select(
        "customer_id", "name", "email", 
        "lifetime_value", "customer_segment", 
        "is_high_value", "processed_at"
    )

# Write to Gold layer
gold_path = "/tmp/medallion/gold/customers"
df_gold.write.format("delta") \
    .mode("overwrite") \
    .save(gold_path)

print("=== Gold Layer: Business-Ready Data ===")
spark.read.format("delta").load(gold_path).show(truncate=False)

# Analytics queries
print("\n=== Customer Segment Distribution ===")
spark.read.format("delta").load(gold_path) \
    .groupBy("customer_segment").count() \
    .show()

print("\n=== Revenue Metrics ===")
spark.read.format("delta").load(gold_path) \
    .agg(
        count("*").alias("total_customers"),
        sum("lifetime_value").alias("total_revenue"),
        avg("lifetime_value").alias("avg_lifetime_value")
    ).show()

---

## Section 3: Performance Tuning

### Key Performance Optimization Techniques

1. **Data Skew**: Uneven data distribution causing performance issues
2. **Salting**: Adding random prefixes to skewed keys
3. **Caching**: Keep frequently-used data in memory
4. **Join Strategies**: Broadcast vs. Sort-Merge joins
5. **Partitioning**: Organize data for efficient queries

### 3.1 Data Skew Problem and Salting Solution

**Data Skew**: When one or few partitions contain much more data than others, some executors work much harder while others are idle.

**Salting**: Adding a random prefix to skewed keys to redistribute data evenly.

In [ ]:
# Create skewed dataset
# Most records belong to 3 customers (skew)
skewed_data = []

# Customer 1: 1000 records
for i in range(1000):
    skewed_data.append((1, f"transaction_{i}", 100.0))

# Customer 2: 800 records
for i in range(800):
    skewed_data.append((2, f"transaction_{i}", 150.0))

# Customer 3: 900 records
for i in range(900):
    skewed_data.append((3, f"transaction_{i}", 120.0))

# Customer 4-10: Few records each
for cust_id in range(4, 11):
    for i in range(10):
        skewed_data.append((cust_id, f"transaction_{i}", 50.0))

skew_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("transaction_id", StringType(), True),
    StructField("amount", DoubleType(), True),
])

df_skewed = spark.createDataFrame(skewed_data, schema=skew_schema)

print("=== Data Skew Analysis ===")
df_skewed.groupBy("customer_id").count().orderBy(col("count").desc()).show()

In [ ]:
# Without salting: GROUP BY will be slow for customers 1, 2, 3
print("=== Without Salting (Single Partition per Customer) ===")
result_without_salt = df_skewed.groupBy("customer_id") \
    .agg(
        count("*").alias("transaction_count"),
        sum("amount").alias("total_amount"),
        avg("amount").alias("avg_amount")
    ).orderBy("customer_id")

result_without_salt.show()
print("\n⚠️  Executors handling customers 1-3 are overloaded")

In [ ]:
# SALTING: Add random prefix to redistribute data
import random

def add_salt(customer_id, num_buckets=10):
    """Add random salt to skewed keys"""
    salt = random.randint(0, num_buckets - 1)
    return f"{customer_id}_{salt}"

# Add salt column
spark.udf.register("add_salt_udf", add_salt)

df_salted = df_skewed.withColumn("salted_customer_id", 
    concat(col("customer_id"), lit("_"), 
           (rand() * 10).cast(IntegerType())))

print("=== With Salting (Data Redistributed) ===")
df_salted.groupBy("salted_customer_id").count().show(10)
print("\n✓ Data is now evenly distributed across 10 buckets")

In [ ]:
# After grouped computation, REMOVE the salt
result_with_salt = df_salted.groupBy("salted_customer_id") \
    .agg(sum("amount").alias("partial_sum")) \
    .withColumn("customer_id", 
        split(col("salted_customer_id"), "_")[0].cast(IntegerType())) \
    .groupBy("customer_id") \
    .agg(sum("partial_sum").alias("total_amount")) \
    .orderBy("customer_id")

print("=== Results After Desalting ===")
result_with_salt.show()
print("\n✓ Computation was parallelized across all executors")

### 3.2 Caching Strategies

Caching keeps frequently-used DataFrames in memory to avoid recomputation.

**When to use:**
- DataFrame used multiple times
- Expensive transformations (joins, aggregations)
- After significant filtering

**Caution:**
- Memory is limited
- Cache invalidated on data changes

In [ ]:
from pyspark import StorageLevel

# Create a expensive-to-compute DataFrame
df_large = spark.read.format("delta").load(gold_path)

# Expensive transformation
df_transformed = df_large \
    .withColumn("revenue_per_transaction", col("lifetime_value") / 100) \
    .withColumn("is_active", col("customer_segment") != "Basic")

print("=== Caching Example ===")

# WITHOUT CACHE: Compute twice
print("\nWithout cache (computes twice):")
result1 = df_transformed.filter(col("is_high_value")).count()
result2 = df_transformed.filter(col("is_active")).count()
print(f"High value customers: {result1}, Active customers: {result2}")

# WITH CACHE: Compute once, reuse from memory
print("\nWith cache (computes once, reuses):")
df_cached = df_transformed.cache()  # or .persist(StorageLevel.MEMORY_ONLY)
result1 = df_cached.filter(col("is_high_value")).count()
result2 = df_cached.filter(col("is_active")).count()
print(f"High value customers: {result1}, Active customers: {result2}")

print("\n✓ Second query is much faster from cache")

# Release cache
df_cached.unpersist()

In [ ]:
# Cache Storage Levels
print("=== Cache Storage Levels ===")
print("""
MEMORY_ONLY: Store in memory, spill to disk if needed
MEMORY_AND_DISK: Store in memory, spill to disk
DISK_ONLY: Store only on disk
MEMORY_ONLY_SER: Serialize in memory (more compact)
MEMORY_AND_DISK_SER: Serialize in memory, spill to disk
""")

# Example with serialization
df_cached_ser = df_transformed.persist(StorageLevel.MEMORY_AND_DISK_SER)
count = df_cached_ser.count()
print(f"Cached {count} rows with serialization")
df_cached_ser.unpersist()

### 3.3 Join Strategies: Broadcast vs. Sort-Merge

**Broadcast Join**: Small table broadcast to all executors (fast for small tables)  
**Sort-Merge Join**: Both tables sorted and joined (for large-large joins)

| Join Type | Table Size | Best For |
|-----------|-----------|----------|
| Broadcast | < 2GB | Dimension * Fact |
| Sort-Merge | > 2GB | Large * Large |

In [ ]:
# Create dimension and fact tables
customers = spark.read.format("delta").load(gold_path).select("customer_id", "name", "customer_segment")

# Create transactions (fact table)
transactions_data = [
    (1, "TX001", 100, 1),
    (1, "TX002", 150, 1),
    (2, "TX003", 200, 2),
    (3, "TX004", 120, 3),
    (4, "TX005", 80, 4),
    (5, "TX006", 160, 5),
]

tx_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("transaction_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("customer_id_2", IntegerType(), True),  # For demo
])

df_transactions = spark.createDataFrame(transactions_data, schema=tx_schema)

print(f"Customers table size: {customers.count()} rows")
print(f"Transactions table size: {df_transactions.count()} rows")

In [ ]:
# BROADCAST JOIN (automatic for small tables)
print("=== Broadcast Join (Small * Large) ===")

result_broadcast = df_transactions.join(
    broadcast(customers),
    df_transactions.customer_id == customers.customer_id
).select(
    df_transactions.transaction_id,
    customers.name,
    df_transactions.amount,
    customers.customer_segment
)

print("\nJoin Plan:")
result_broadcast.explain()
print("\n✓ BroadcastHashJoin is efficient for small tables")

In [ ]:
# Force Sort-Merge Join (for comparison)
print("=== Sort-Merge Join (Large * Large) ===")

result_sort_merge = df_transactions.join(
    customers,
    df_transactions.customer_id == customers.customer_id,
    "inner"
).select(
    df_transactions.transaction_id,
    customers.name,
    df_transactions.amount,
    customers.customer_segment
)

print("\nJoin Plan:")
result_sort_merge.explain()
print("\n✓ SortMergeJoin used when broadcast not suitable")

In [ ]:
# Check execution
print("=== Join Results ===")
result_broadcast.show()

---

## Section 4: Databricks Features

### 4.1 Unity Catalog: Data Governance

Unity Catalog is Databricks' governance framework providing:
- **Centralized access control** across workspaces
- **Data lineage** tracking
- **Tagging** for data classification
- **Audit logs** for compliance

**Hierarchy**: Metastore → Catalog → Schema → Table

In [ ]:
# In a real Databricks workspace with Unity Catalog enabled:

# Check catalog structure
print("=== Unity Catalog Concepts ===")
print("""
METASTORE: Root-level container (typically 1 per account)
├─ CATALOG: Logical grouping (e.g., "analytics", "finance")
│  ├─ SCHEMA: Database within catalog
│  │  ├─ TABLE: Data table
│  │  ├─ VIEW: Query view
│  │  └─ VOLUME: External data container

Example:
metastore_default
├─ main (built-in catalog)
│  ├─ default (default schema)
│  ├─ reporting (schema)
│  │  ├─ customers (table)
│  │  └─ sales (table)
├─ analytics (custom catalog)
│  └─ gold (schema)
""")

# Try to list catalogs
try:
    catalogs = spark.sql("SHOW CATALOGS").collect()
    print(f"\nAvailable catalogs: {catalogs}")
except:
    print("\nNote: Unity Catalog not enabled in this workspace")
    print("In Databricks with UC enabled, use: SHOW CATALOGS")

In [ ]:
# Working with schemas (databases)
print("=== Working with Schemas ===")

# Show default schemas
spark.sql("SHOW DATABASES").show()

# Create a schema (if not exists)
spark.sql("CREATE SCHEMA IF NOT EXISTS masterclass_demo")

# Use the schema
spark.sql("USE masterclass_demo")
print("✓ Schema 'masterclass_demo' created and selected")

In [ ]:
# Tagging and Metadata
print("=== Data Tagging and Metadata ===")

# Create a tagged table
df_tagged = spark.read.format("delta").load(gold_path)

# In Databricks, use: ALTER TABLE ... SET TBLPROPERTIES
# For this demo, we'll show the concept

# Write with metadata
df_tagged.write.format("delta") \
    .mode("overwrite") \
    .option("comment", "Customer analytics table - PII data") \
    .save("/tmp/tagged_customers")

print("✓ Table metadata set")
print("")
print("Common tags:")
print("- PII: Contains personally identifiable information")
print("- SENSITIVE: Requires access control")
print("- PROD: Production data")
print("- CONFIDENTIAL: Restricted access")

### 4.2 Delta Live Tables (DLT): Declarative Data Pipelines

Delta Live Tables is a framework for building reliable data pipelines using SQL or Python.

**Key Concepts:**
- **Declarative**: Describe data, not transformation steps
- **Automatic Optimization**: DLT handles caching, multi-hop, retries
- **Quality Monitoring**: Built-in data quality checks
- **Lineage Tracking**: Automatic data lineage

In [ ]:
# Delta Live Tables syntax (shown for reference - runs in DLT context)
print("=== Delta Live Tables (DLT) Pattern ===")

dlt_code = '''
import dlt
from pyspark.sql.functions import *

# Bronze layer: ingestion
@dlt.table(
    name="customers_bronze",
    comment="Raw customer data from sources",
    table_properties={"quality": "bronze"}
)
def customers_bronze():
    return spark.read.format("delta").load("/path/to/raw/data")

# Silver layer: quality checks
@dlt.table(
    name="customers_silver",
    comment="Cleaned customer data"
)
@dlt.expect("valid_email", "email LIKE '%@%'")
@dlt.expect("positive_value", "lifetime_value > 0")
def customers_silver():
    return dlt.read("customers_bronze") \\
        .filter(col("name").isNotNull()) \\
        .dropDuplicates(["customer_id"])

# Gold layer: aggregations
@dlt.table(
    name="customer_metrics",
    comment="Customer analytics metrics"
)
def customer_metrics():
    return dlt.read("customers_silver") \\
        .withColumn("segment", 
            when(col("lifetime_value") >= 200000, "Premium") \\
            .otherwise("Standard")) \\
        .groupBy("segment") \\
        .agg(count("*").alias("count"), 
             avg("lifetime_value").alias("avg_value"))
'''

print(dlt_code)
print("\nKey advantages:")
print("✓ Automatic scheduling and error handling")
print("✓ Data quality expectations built-in")
print("✓ Complete lineage tracking")
print("✓ Simplified dependency management")

### 4.3 Delta Sharing: Secure Data Collaboration

Delta Sharing enables secure, real-time data sharing across organizations without copying data.

**Features:**
- Share data without moving or duplicating
- Open standard (works with non-Databricks tools)
- Fine-grained access control
- Real-time updates

In [ ]:
# Delta Sharing concepts
print("=== Delta Sharing Architecture ===")

sharing_concept = '''
PROVIDER (Data Owner)
├─ Create Share
├─ Create Recipient
├─ Grant permissions on tables
└─ Share with recipient via credentials

RECIPIENT (Data Consumer)
├─ Receive credentials
├─ Add to catalog (add_share)
├─ Query tables in real-time
└─ No data copy needed

Example workflow:
1. Provider creates share: CREATE SHARE my_data_share
2. Provider grants table: ALTER SHARE my_data_share ADD TABLE my_catalog.my_schema.my_table
3. Provider creates recipient: CREATE RECIPIENT my_partner
4. Provider creates credentials for sharing
5. Recipient adds share using credentials
6. Recipient queries: SELECT * FROM shared_data.my_schema.my_table
'''

print(sharing_concept)

print("\nUse cases:")
print("• B2B data partnerships")
print("• Multi-team collaboration")
print("• Vendor data provision")
print("• Regulatory compliance sharing")

---

## Section 5: PySpark Mastery

### 5.1 DataFrame API: Core Operations

The DataFrame API is the high-level API for working with structured data in Spark.

In [ ]:
# Load sample data
df = spark.read.format("delta").load(gold_path)

print("=== DataFrame Operations Cheat Sheet ===")
print(f"Total rows: {df.count()}")
print(f"Total columns: {len(df.columns)}")
print(f"\nColumns: {df.columns}")

In [ ]:
# SELECT (projection)
print("=== SELECT: Choose columns ===")
df.select("customer_id", "name", "customer_segment").show()

# SELECT with expressions
print("\n=== SELECT with Expressions ===")
df.selectExpr("customer_id", "name as customer_name", "lifetime_value * 2 as doubled_value").show()

# Rename columns
print("\n=== RENAME: withColumnRenamed ===")
df.withColumnRenamed("lifetime_value", "ltv").select("customer_id", "ltv").show()

In [ ]:
# FILTER (WHERE clause)
print("=== FILTER: Rows that match condition ===")
df.filter(col("lifetime_value") > 175000).select("name", "lifetime_value").show()

# Multiple conditions
print("\n=== Multiple Conditions ===")
df.filter((col("lifetime_value") > 150000) & (col("customer_segment") == "Premium")).show()

# SQL syntax
print("\n=== Using SQL Syntax ===")
df.filter("lifetime_value > 175000 AND customer_segment = 'Premium'").show()

In [ ]:
# GROUP BY with aggregations
print("=== GROUP BY: Aggregations ===")
df.groupBy("customer_segment").agg(
    count("*").alias("count"),
    sum("lifetime_value").alias("total_value"),
    avg("lifetime_value").alias("avg_value"),
    min("lifetime_value").alias("min_value"),
    max("lifetime_value").alias("max_value")
).show()

# Multiple group columns
print("\n=== GROUP BY Multiple Columns ===")
df.groupBy("customer_segment", "is_high_value").count().show()

In [ ]:
# WINDOW functions
from pyspark.sql.window import Window

print("=== WINDOW Functions: Ranking, Running Totals ===")

# Rank within segment
window_spec = Window.partitionBy("customer_segment").orderBy(col("lifetime_value").desc())

df_ranked = df.withColumn(
    "rank_in_segment",
    rank().over(window_spec)
).withColumn(
    "dense_rank",
    dense_rank().over(window_spec)
).select(
    "customer_segment", "name", "lifetime_value", "rank_in_segment", "dense_rank"
)

df_ranked.show()

print("\n✓ rank(): Allows gaps | dense_rank(): No gaps")

In [ ]:
# Joins
print("=== JOINs: Combine DataFrames ===")

# Create a second DataFrame
orders_data = [
    (1, "ORD001", 100),
    (1, "ORD002", 150),
    (2, "ORD003", 200),
    (5, "ORD004", 120),
]

orders_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("order_id", StringType(), True),
    StructField("order_value", DoubleType(), True),
])

df_orders = spark.createDataFrame(orders_data, schema=orders_schema)

# INNER JOIN
print("\nINNER JOIN:")
df.join(df_orders, "customer_id", "inner") \
    .select("customer_id", "name", "order_id", "order_value") \
    .show()

# LEFT JOIN
print("\nLEFT JOIN (all customers, matched orders if available):")
df.join(df_orders, "customer_id", "left") \
    .select("customer_id", "name", "order_id", "order_value") \
    .show()

### 5.2 User Defined Functions (UDFs)

UDFs allow you to define custom transformation logic that works on each row.

In [ ]:
# Python UDF
print("=== Python UDFs ===")

# Define a Python function
def categorize_customer(ltv):
    if ltv is None:
        return "Unknown"
    elif ltv >= 250000:
        return "Platinum"
    elif ltv >= 200000:
        return "Gold"
    elif ltv >= 150000:
        return "Silver"
    else:
        return "Bronze"

# Register as UDF
categorize_customer_udf = udf(categorize_customer, StringType())

# Use UDF
df_with_category = df.withColumn(
    "customer_category",
    categorize_customer_udf(col("lifetime_value"))
)

df_with_category.select("name", "lifetime_value", "customer_category").show()

In [ ]:
# Pandas UDF (Vectorized) - Better Performance
print("=== Pandas UDFs (Vectorized) ===")
import pandas as pd
from pyspark.sql.functions import pandas_udf

# Pandas UDF processes batches of data
@pandas_udf(StringType())
def categorize_customer_pandas(ltv):
    # ltv is a pandas Series
    return ltv.apply(
        lambda x: "Platinum" if x >= 250000 else 
                 ("Gold" if x >= 200000 else 
                 ("Silver" if x >= 150000 else "Bronze"))
    )

df_with_category_pandas = df.withColumn(
    "customer_category_fast",
    categorize_customer_pandas(col("lifetime_value"))
)

df_with_category_pandas.select("name", "lifetime_value", "customer_category_fast").show()

print("\n✓ Pandas UDFs are much faster than Python UDFs (vectorized processing)")

In [ ]:
# Register UDF for SQL use
print("=== Registering UDFs for SQL ===")

spark.udf.register("categorize_sql", categorize_customer_udf)

# Now use in SQL
df.createOrReplaceTempView("customers_view")

result = spark.sql("""
    SELECT 
        customer_id,
        name,
        lifetime_value,
        categorize_sql(lifetime_value) as category
    FROM customers_view
    ORDER BY lifetime_value DESC
""")

result.show()

### 5.3 Handling JSON and Nested Schemas

Real-world data often comes in nested formats (JSON, nested structs).

In [ ]:
# Create nested JSON data
print("=== Nested Schema Example ===")

nested_data = [
    (1, "Alice", {"city": "NYC", "state": "NY", "zip": "10001"}, ["email", "phone"]),
    (2, "Bob", {"city": "LA", "state": "CA", "zip": "90001"}, ["email", "sms"]),
    (3, "Carol", {"city": "Chicago", "state": "IL", "zip": "60601"}, ["email", "phone", "push"]),
]

nested_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("address", StructType([
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("zip", StringType(), True),
    ]), True),
    StructField("contact_methods", ArrayType(StringType()), True),
])

df_nested = spark.createDataFrame(nested_data, schema=nested_schema)

print("\nSchema:")
df_nested.printSchema()

print("\nData:")
df_nested.show(truncate=False)

In [ ]:
# Access nested fields
print("=== Accessing Nested Fields ===")

# Access struct fields
df_unnested = df_nested.select(
    "customer_id",
    "name",
    col("address.city").alias("city"),
    col("address.state").alias("state"),
    col("address.zip").alias("zip")
)

print("\nAccessed struct fields:")
df_unnested.show()

In [ ]:
# Explode array field
print("=== Exploding Array Fields ===")

df_exploded = df_nested.select(
    "customer_id",
    "name",
    col("address.city").alias("city"),
    explode(col("contact_methods")).alias("contact_method")
)

print("\nOne row per contact method:")
df_exploded.show()

print("\n✓ Each array element becomes a separate row")

In [ ]:
# Parse JSON string
print("=== Parsing JSON Strings ===")

json_data = [
    (1, '{"city": "NYC", "state": "NY"}'),
    (2, '{"city": "LA", "state": "CA"}'),
]

json_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("address_json", StringType(), True),
])

df_json_str = spark.createDataFrame(json_data, schema=json_schema)

# Parse JSON
df_parsed = df_json_str.withColumn(
    "address",
    from_json(col("address_json"), StructType([
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
    ]))
).select(
    "customer_id",
    col("address.city").alias("city"),
    col("address.state").alias("state")
)

print("\nParsed from JSON:")
df_parsed.show()

---

## Section 6: Advanced Optimization

### 6.1 Z-Ordering: Multi-Dimensional Clustering

Z-Ordering automatically sorts data on multiple columns to improve query performance for multi-dimensional filters.

In [ ]:
# Create a larger dataset for Z-Order demo
print("=== Z-Ordering Demo ===")

# Create sample data
zorder_data = []
for i in range(1, 101):
    zorder_data.append((
        i,
        f"Customer_{i}",
        "NYC" if i % 3 == 0 else ("LA" if i % 3 == 1 else "Chicago"),
        "Premium" if i % 2 == 0 else "Standard",
        100000 + (i * 1000)
    ))

zorder_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("lifetime_value", LongType(), True),
])

df_zorder = spark.createDataFrame(zorder_data, schema=zorder_schema)

# Save for Z-ordering
zorder_path = "/tmp/zorder_demo"
df_zorder.write.format("delta") \
    .mode("overwrite") \
    .save(zorder_path)

print(f"✓ Data written to {zorder_path}")

In [ ]:
# Apply Z-ordering
print("=== Applying Z-Order ===")
print("Z-Order on columns: city, segment, lifetime_value")
print("This optimizes queries filtering on these columns...\n")

spark.sql(f"OPTIMIZE delta.`{zorder_path}` ZORDER BY (city, segment, lifetime_value)")

print("✓ Z-Ordering complete")
print("\nBenefit: Queries filtering on (city, segment, lifetime_value) are much faster")

In [ ]:
# Query after Z-ordering
print("=== Query After Z-Ordering ===")

result = spark.read.format("delta").load(zorder_path).filter(
    (col("city") == "NYC") & 
    (col("segment") == "Premium") & 
    (col("lifetime_value") > 150000)
)

print(f"Found {result.count()} Premium customers in NYC with LTV > 150000")
result.show()

### 6.2 Data Skipping: Automatic Index

Delta Lake maintains statistics (min/max, count of nulls) on data files. Data skipping automatically prunes files that can't match query filters.

In [ ]:
# View data skipping statistics
print("=== Data Skipping Statistics ===")

stats_df = spark.sql(f"DESCRIBE DETAIL delta.`{zorder_path}`")
stats_df.show(truncate=False)

print("\nDelta maintains min/max for each column on each file")
print("When you query for city = 'NYC', files without NYC data are skipped")
print("This dramatically speeds up queries on large tables")

In [ ]:
# View file statistics
print("=== File-Level Statistics ===")

# In Databricks, use: SELECT * FROM delta.`path`._symlink_format_manifest/
# For now, show concept:

print("""  
Delta maintains statistics like:
- min_value, max_value for each column
- null_count for each column
- row_count for the file

Example file stats:
File: part-00000.parquet
  city: min="Chicago", max="NYC", nulls=0
  segment: min="Premium", max="Standard", nulls=0
  lifetime_value: min=100000, max=150000, nulls=0

Query: WHERE city = 'LA' AND lifetime_value > 120000
  ✓ File matches (city range includes potential 'LA')
  ✓ File matches (lifetime_value max 150000 > 120000)
  
Query: WHERE city = 'LA' AND lifetime_value > 200000
  ✗ File skipped! (lifetime_value max 150000 < 200000)
""")

print("✓ Data skipping eliminates scanning unnecessary files")

---

## Section 7: End-to-End Workloads

### Complete Data Engineering Pipeline

This section demonstrates a realistic data engineering workload combining all concepts:
- Ingest from multiple sources (Bronze)
- Clean and consolidate (Silver)
- Aggregate for analytics (Gold)
- Optimize for performance

### Workload 1: Daily Customer Analytics Pipeline

**Scenario**: Ingest customer data from 3 sources daily, clean it, and produce analytics

In [ ]:
import random

print("=== WORKLOAD 1: Daily Customer Analytics Pipeline ===")
print("\nStep 1: INGEST - Bronze Layer")
print("Source 1: Database export")
print("Source 2: API endpoint")
print("Source 3: CSV file")

# Simulate 3 data sources
def create_source1_data(count=20):
    """Database export data"""
    data = []
    for i in range(count):
        data.append((
            1000 + i,
            f"Customer_{i}",
            f"customer_{i}@company.com",
            random.choice(["NYC", "LA", "Chicago"]),
            random.randint(50000, 500000),
            "db_export",
            current_timestamp()
        ))
    return data

def create_source2_data(count=15):
    """API data"""
    data = []
    for i in range(count):
        data.append((
            2000 + i,
            f"APIUser_{i}",
            f"apiuser_{i}@company.com",
            random.choice(["NYC", "LA", "Chicago", "Denver"]),
            random.randint(50000, 500000),
            "api",
            current_timestamp()
        ))
    return data

def create_source3_data(count=10):
    """CSV data"""
    data = []
    for i in range(count):
        data.append((
            3000 + i,
            f"CSVUser_{i}",
            f"csvuser_{i}@company.com" if i % 5 != 0 else None,  # Simulate missing emails
            random.choice(["NYC", "LA", "Chicago"]),
            random.randint(50000, 500000),
            "csv",
            current_timestamp()
        ))
    return data

# Create DataFrames
bronze_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("annual_revenue", IntegerType(), True),
    StructField("source", StringType(), True),
    StructField("ingestion_time", TimestampType(), True),
])

df_source1 = spark.createDataFrame(create_source1_data(), schema=bronze_schema)
df_source2 = spark.createDataFrame(create_source2_data(), schema=bronze_schema)
df_source3 = spark.createDataFrame(create_source3_data(), schema=bronze_schema)

# Combine all sources
df_bronze_combined = df_source1.union(df_source2).union(df_source3)

# Write to Bronze
bronze_workload_path = "/tmp/workload1/bronze/customers"
df_bronze_combined.write.format("delta").mode("overwrite").save(bronze_workload_path)

print(f"\n✓ Ingested {df_bronze_combined.count()} records from 3 sources")
print(f"  - {df_source1.count()} from Database")
print(f"  - {df_source2.count()} from API")
print(f"  - {df_source3.count()} from CSV")

In [ ]:
print("\nStep 2: CLEAN - Silver Layer")

df_bronze = spark.read.format("delta").load(bronze_workload_path)

# Quality checks before cleaning
print(f"\nBronze Data Quality Checks:")
print(f"  Total records: {df_bronze.count()}")
print(f"  Null emails: {df_bronze.filter(col('email').isNull()).count()}")
print(f"  Duplicate customer_ids: {df_bronze.count() - df_bronze.select('customer_id').distinct().count()}")
print(f"  Zero/negative revenue: {df_bronze.filter(col('annual_revenue') <= 0).count()}")

# Clean data
df_silver = df_bronze \
    .filter(col("email").isNotNull()) \
    .filter(col("annual_revenue") > 0) \
    .dropDuplicates(["customer_id"]) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("name", initcap(trim(col("name")))) \
    .withColumn("processed_at", current_timestamp())

# Write to Silver
silver_workload_path = "/tmp/workload1/silver/customers"
df_silver.write.format("delta").mode("overwrite").save(silver_workload_path)

print(f"\n✓ Cleaned {df_silver.count()} records after quality filters")

In [ ]:
print("\nStep 3: AGGREGATE - Gold Layer")

df_silver = spark.read.format("delta").load(silver_workload_path)

# Create multiple gold layer tables

# 1. Customer dimension
print("\n1. Customer Dimension Table:")
df_customer_dim = df_silver.select(
    "customer_id", "name", "email", "city",
    "annual_revenue", "processed_at"
)

gold_customer_path = "/tmp/workload1/gold/customer_dim"
df_customer_dim.write.format("delta").mode("overwrite").save(gold_customer_path)
print(f"  ✓ {df_customer_dim.count()} unique customers")

# 2. Revenue by city
print("\n2. Revenue by City (Fact Table):")
df_revenue_by_city = df_silver.groupBy("city").agg(
    count("*").alias("customer_count"),
    sum("annual_revenue").alias("total_revenue"),
    avg("annual_revenue").alias("avg_revenue"),
    min("annual_revenue").alias("min_revenue"),
    max("annual_revenue").alias("max_revenue")
).withColumn("calculated_at", current_timestamp())

gold_city_path = "/tmp/workload1/gold/revenue_by_city"
df_revenue_by_city.write.format("delta").mode("overwrite").save(gold_city_path)

df_revenue_by_city.show()

# 3. Customer segmentation
print("\n3. Customer Segmentation:")
df_segments = df_silver \
    .withColumn("revenue_segment",
        when(col("annual_revenue") >= 400000, "High")
        .when(col("annual_revenue") >= 200000, "Medium")
        .otherwise("Low")
    ) \
    .groupBy("revenue_segment", "city").agg(
        count("*").alias("count"),
        avg("annual_revenue").alias("avg_revenue")
    )

gold_segments_path = "/tmp/workload1/gold/customer_segments"
df_segments.write.format("delta").mode("overwrite").save(gold_segments_path)

df_segments.show()

print(f"\n✓ Gold layer tables created successfully")

In [ ]:
print("\nStep 4: OPTIMIZE - Performance Tuning")

# Optimize Silver layer
print(f"\nOptimizing {silver_workload_path}...")
spark.sql(f"OPTIMIZE delta.`{silver_workload_path}`")

# Z-Order Gold layer for multi-dimensional queries
print(f"\nZ-Ordering {gold_city_path}...")
spark.sql(f"OPTIMIZE delta.`{gold_city_path}` ZORDER BY (city)")

print("\n✓ Optimization complete")
print("\nSummary:")
print(f"  Bronze: {bronze_workload_path}")
print(f"  Silver: {silver_workload_path}")
print(f"  Gold: ")
print(f"    - {gold_customer_path}")
print(f"    - {gold_city_path}")
print(f"    - {gold_segments_path}")

In [ ]:
print("\nStep 5: VALIDATE - Data Quality and Testing")

# Check data quality
df_gold = spark.read.format("delta").load(gold_customer_path)

print("\nGold Layer Validation:")
print(f"  Total customers: {df_gold.count()}")
print(f"  Null customer_ids: {df_gold.filter(col('customer_id').isNull()).count()}")
print(f"  Null emails: {df_gold.filter(col('email').isNull()).count()}")
print(f"  Revenue range: ${df_gold.agg(min('annual_revenue')).collect()[0][0]} - ${df_gold.agg(max('annual_revenue')).collect()[0][0]}")

print("\n✓ All validation checks passed")

### Workload 2: Real-Time Streaming Analytics

**Scenario**: Stream events to Delta Lake and compute rolling analytics

In [ ]:
print("=== WORKLOAD 2: Streaming Analytics (Batch Simulation) ===")
print("\nIn production, this uses Spark Structured Streaming with Delta Lake")
print("For demo, we'll simulate batches arriving over time\n")

# Simulate streaming events
def generate_event_batch(batch_num):
    """Simulate incoming event batch"""
    events = []
    for i in range(random.randint(50, 100)):
        events.append((
            random.randint(1000, 3010),  # customer_id
            f"purchase_{batch_num}_{i}",  # event_id
            random.choice(["Purchase", "View", "AddToCart", "Checkout"]),  # event_type
            random.randint(10, 500),  # value
            current_timestamp()  # timestamp
        ))
    return events

# Create event schema
event_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("event_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("event_time", TimestampType(), True),
])

# Append batches to Delta table (simulating streaming)
stream_path = "/tmp/workload2/events"

for batch in range(3):
    print(f"Batch {batch + 1}: Processing events...")
    batch_data = generate_event_batch(batch)
    df_batch = spark.createDataFrame(batch_data, schema=event_schema)
    
    mode = "overwrite" if batch == 0 else "append"
    df_batch.write.format("delta").mode(mode).save(stream_path)
    print(f"  ✓ {len(batch_data)} events written")

print(f"\n✓ Total events processed: {spark.read.format('delta').load(stream_path).count()}")

In [ ]:
# Compute streaming analytics
print("\nStreaming Analytics - Real-time Metrics:")

df_events = spark.read.format("delta").load(stream_path)

# 1. Event counts by type
print("\n1. Events by Type:")
df_events.groupBy("event_type").count().orderBy(col("count").desc()).show()

# 2. Revenue by customer
print("\n2. Top 10 Customers by Revenue:")
df_events.filter(col("event_type") == "Purchase") \
    .groupBy("customer_id") \
    .agg(sum("value").alias("total_spent"), count("*").alias("purchase_count")) \
    .orderBy(col("total_spent").desc()) \
    .limit(10) \
    .show()

# 3. Conversion metric
print("\n3. Conversion Funnel:")
df_events.groupBy("event_type").count().show()